In [1]:
from pyspark.sql import SparkSession
import os 

os.environ["HADOOP_USER_NAME"] = "root"

spark = SparkSession.builder.appName("transformation").master("local[*]")\
.config("spark.hadoop.fs.defaultFS", "hdfs://hadoop-namenode:9000")\
.getOrCreate()

In [2]:
bronze_path = "hdfs://hadoop-namenode:9000/user/root/datalake/bronze/power_consumption"

bronze_df = spark.read.parquet(bronze_path)

bronze_df.printSchema()
bronze_df.show(5)
print(bronze_df.count())


root
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- Global_active_power: string (nullable = true)
 |-- Global_reactive_power: string (nullable = true)
 |-- Voltage: string (nullable = true)
 |-- Global_intensity: string (nullable = true)
 |-- Sub_metering_1: string (nullable = true)
 |-- Sub_metering_2: string (nullable = true)
 |-- Sub_metering_3: double (nullable = true)

+----------+-------------------+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+
|      Date|               Time|Global_active_power|Global_reactive_power|Voltage|Global_intensity|Sub_metering_1|Sub_metering_2|Sub_metering_3|
+----------+-------------------+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+
|29/11/2008|2026-04-25 14:44:00|              1.498|                0.082|243.250|           6.000|         1.000|         0.000|          19.0|
|29/11/200

In [34]:
bronze_df.printSchema()
bronze_df.describe().show()

root
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- Global_active_power: string (nullable = true)
 |-- Global_reactive_power: string (nullable = true)
 |-- Voltage: string (nullable = true)
 |-- Global_intensity: string (nullable = true)
 |-- Sub_metering_1: string (nullable = true)
 |-- Sub_metering_2: string (nullable = true)
 |-- Sub_metering_3: double (nullable = true)

+-------+--------+-------------------+---------------------+------------------+------------------+------------------+------------------+-----------------+
|summary|    Date|Global_active_power|Global_reactive_power|           Voltage|  Global_intensity|    Sub_metering_1|    Sub_metering_2|   Sub_metering_3|
+-------+--------+-------------------+---------------------+------------------+------------------+------------------+------------------+-----------------+
|  count| 2075259|            2075259|              2075259|           2075259|           2075259|           2075259|        

In [4]:
from pyspark.sql import functions as F

bronze_df.select([
    F.count(F.when(F.col(c).isNull(),c)).alias(c)
    for c in bronze_df.columns
]).show()

+----+----+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+
|Date|Time|Global_active_power|Global_reactive_power|Voltage|Global_intensity|Sub_metering_1|Sub_metering_2|Sub_metering_3|
+----+----+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+
|   0|   0|                  0|                    0|      0|               0|             0|             0|         25979|
+----+----+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+



In [5]:
silver_df = bronze_df.fillna({
    "Sub_metering_3" : 0
})
silver_df.select("Sub_metering_3").show(5)

+--------------+
|Sub_metering_3|
+--------------+
|          19.0|
|          18.0|
|          18.0|
|          18.0|
|          18.0|
+--------------+
only showing top 5 rows



In [6]:
from pyspark.sql import functions as F

silver_df = silver_df.withColumn(
    "Time_fixed",
    F.date_format(F.col("time"), "HH:mm:ss")
)

silver_df = silver_df.withColumn(
    "datetime_str",
    F.concat_ws(" ", F.col("Date"), F.col("Time_fixed"))
)

silver_df = silver_df.withColumn(
    "event_timestamp",
    F.to_timestamp(F.col("datetime_str"), "d/M/yyyy HH:mm:ss")
)

silver_df.select("Date", "Time_fixed", "datetime_str", "event_timestamp").show(5, truncate=False)

+----------+----------+-------------------+-------------------+
|Date      |Time_fixed|datetime_str       |event_timestamp    |
+----------+----------+-------------------+-------------------+
|29/11/2008|14:44:00  |29/11/2008 14:44:00|2008-11-29 14:44:00|
|29/11/2008|14:45:00  |29/11/2008 14:45:00|2008-11-29 14:45:00|
|29/11/2008|14:46:00  |29/11/2008 14:46:00|2008-11-29 14:46:00|
|29/11/2008|14:47:00  |29/11/2008 14:47:00|2008-11-29 14:47:00|
|29/11/2008|14:48:00  |29/11/2008 14:48:00|2008-11-29 14:48:00|
+----------+----------+-------------------+-------------------+
only showing top 5 rows



In [7]:
silver_df = silver_df.replace("?", None)

numeric_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

for col in numeric_cols:
    silver_df = silver_df.withColumn(col, F.col(col).cast("double"))

silver_df.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- Global_active_power: double (nullable = true)
 |-- Global_reactive_power: double (nullable = true)
 |-- Voltage: double (nullable = true)
 |-- Global_intensity: double (nullable = true)
 |-- Sub_metering_1: double (nullable = true)
 |-- Sub_metering_2: double (nullable = true)
 |-- Sub_metering_3: double (nullable = false)
 |-- Time_fixed: string (nullable = true)
 |-- datetime_str: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)



In [8]:
silver_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in silver_df.columns
]).show()

+----+----+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+----------+------------+---------------+
|Date|Time|Global_active_power|Global_reactive_power|Voltage|Global_intensity|Sub_metering_1|Sub_metering_2|Sub_metering_3|Time_fixed|datetime_str|event_timestamp|
+----+----+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+----------+------------+---------------+
|   0|   0|              25979|                25979|  25979|           25979|         25979|         25979|             0|         0|           0|              0|
+----+----+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+----------+------------+---------------+



In [9]:
silver_df = silver_df.dropna(subset=[
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
    "event_timestamp"
])

silver_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in silver_df.columns
]).show()

+----+----+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+----------+------------+---------------+
|Date|Time|Global_active_power|Global_reactive_power|Voltage|Global_intensity|Sub_metering_1|Sub_metering_2|Sub_metering_3|Time_fixed|datetime_str|event_timestamp|
+----+----+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+----------+------------+---------------+
|   0|   0|                  0|                    0|      0|               0|             0|             0|             0|         0|           0|              0|
+----+----+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+----------+------------+---------------+



In [11]:
silver_df = silver_df \
    .withColumn("hour", F.hour("event_timestamp")) \
    .withColumn("day", F.dayofmonth("event_timestamp")) \
    .withColumn("month", F.month("event_timestamp")) \
    .withColumn("year", F.year("event_timestamp")) \
    .withColumn("day_of_week", F.dayofweek("event_timestamp"))

silver_df = silver_df.withColumn(
    "total_sub_metering",
    F.col("Sub_metering_1") + F.col("Sub_metering_2") + F.col("Sub_metering_3")
)

silver_df = silver_df.drop("Date", "Time", "Time_fixed", "datetime_str")

silver_df.printSchema()
silver_df.show(5, truncate=False)
print(silver_df.count())

root
 |-- Global_active_power: double (nullable = true)
 |-- Global_reactive_power: double (nullable = true)
 |-- Voltage: double (nullable = true)
 |-- Global_intensity: double (nullable = true)
 |-- Sub_metering_1: double (nullable = true)
 |-- Sub_metering_2: double (nullable = true)
 |-- Sub_metering_3: double (nullable = false)
 |-- event_timestamp: timestamp (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- total_sub_metering: double (nullable = true)

+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+-------------------+----+---+-----+----+-----------+------------------+
|Global_active_power|Global_reactive_power|Voltage|Global_intensity|Sub_metering_1|Sub_metering_2|Sub_metering_3|event_timestamp    |hour|day|month|year|day_of_week|total_sub_metering|
+-

In [14]:
silver_df.select([
    F.count(F.when(F.col(c).isNull(),c)).alias(c)
    for c in silver_df.columns
]).show()

+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+---------------+----+---+-----+----+-----------+------------------+
|Global_active_power|Global_reactive_power|Voltage|Global_intensity|Sub_metering_1|Sub_metering_2|Sub_metering_3|event_timestamp|hour|day|month|year|day_of_week|total_sub_metering|
+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+---------------+----+---+-----+----+-----------+------------------+
|                  0|                    0|      0|               0|             0|             0|             0|              0|   0|  0|    0|   0|          0|                 0|
+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+---------------+----+---+-----+----+-----------+------------------+



In [16]:
silver_df.select(F.col('total_sub_metering')).show(10)

+------------------+
|total_sub_metering|
+------------------+
|              20.0|
|              19.0|
|              20.0|
|              19.0|
|              19.0|
|              20.0|
|              22.0|
|              21.0|
|              19.0|
|              19.0|
+------------------+
only showing top 10 rows



In [47]:
silver_df.select("Global_active_power","Voltage","day").show()

+-------------------+-------+---+
|Global_active_power|Voltage|day|
+-------------------+-------+---+
|              1.498| 243.25| 29|
|              1.518| 243.37| 29|
|              1.502| 243.89| 29|
|              1.518|  243.8| 29|
|              1.492| 243.36| 29|
|              1.542| 242.58| 29|
|              1.586| 242.32| 29|
|              1.566| 242.35| 29|
|              1.532| 242.42| 29|
|              1.574| 242.76| 29|
|              1.546| 242.23| 29|
|               1.57| 242.55| 29|
|              1.546| 242.51| 29|
|              1.566| 242.81| 29|
|              1.548| 243.26| 29|
|              1.568|  243.1| 29|
|              1.554| 243.82| 29|
|              1.564|  243.0| 29|
|              1.548| 243.44| 29|
|              1.566| 243.58| 29|
+-------------------+-------+---+
only showing top 20 rows



In [48]:
silver_df = silver_df.withColumn(
    "is_weekend",
    F.when(F.col("day_of_week").isin(1, 7), 1).otherwise(0)
)

In [82]:
silver_df = silver_df.withColumn("peak_hours",
                     F.when((F.col('hour')>=18) & (F.col('hour')<=23),1).otherwise(0))

In [85]:
silver_df.select(
    "event_timestamp",
    "hour",
    "day",
    "month",
    "day_of_week",
    "is_weekend",
    "peak_hours"
).show(50, False)

+-------------------+----+---+-----+-----------+----------+----------+
|event_timestamp    |hour|day|month|day_of_week|is_weekend|peak_hours|
+-------------------+----+---+-----+-----------+----------+----------+
|2008-11-29 14:44:00|14  |29 |11   |7          |1         |0         |
|2008-11-29 14:45:00|14  |29 |11   |7          |1         |0         |
|2008-11-29 14:46:00|14  |29 |11   |7          |1         |0         |
|2008-11-29 14:47:00|14  |29 |11   |7          |1         |0         |
|2008-11-29 14:48:00|14  |29 |11   |7          |1         |0         |
|2008-11-29 14:49:00|14  |29 |11   |7          |1         |0         |
|2008-11-29 14:50:00|14  |29 |11   |7          |1         |0         |
|2008-11-29 14:51:00|14  |29 |11   |7          |1         |0         |
|2008-11-29 14:52:00|14  |29 |11   |7          |1         |0         |
|2008-11-29 14:53:00|14  |29 |11   |7          |1         |0         |
|2008-11-29 14:54:00|14  |29 |11   |7          |1         |0         |
|2008-

In [86]:
silver_path = "hdfs://hadoop-namenode:9000/user/root/datalake/silver/power_consumption"

silver_df.write \
    .mode("overwrite") \
    .parquet(silver_path)



In [87]:
check_silver = spark.read.parquet(silver_path)
check_silver.show(5)
print(check_silver.count())

+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+-------------------+----+---+-----+----+-----------+------------------+----------+----------+
|Global_active_power|Global_reactive_power|Voltage|Global_intensity|Sub_metering_1|Sub_metering_2|Sub_metering_3|    event_timestamp|hour|day|month|year|day_of_week|total_sub_metering|is_weekend|peak_hours|
+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+-------------------+----+---+-----+----+-----------+------------------+----------+----------+
|              1.498|                0.082| 243.25|             6.0|           1.0|           0.0|          19.0|2008-11-29 14:44:00|  14| 29|   11|2008|          7|              20.0|         1|         0|
|              1.518|                0.088| 243.37|             6.2|           1.0|           0.0|          18.0|2008-11-29 14:45:00|  14| 29|   11|2008|          7|       